# reduce-gather-sum — faded example 3: Pre-allocate the gather list before all_gather

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-gather-sum`. The last cell reports your progress on the `Distributed: reduce.gather + sum` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce.gather + sum` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-gather-sum`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-gather-sum"
DD_SUBTOPIC = "Distributed: reduce.gather + sum"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`all_gather` does not allocate its output: the caller must pass a list of `world_size` pre-allocated tensors, each the same shape as the local tensor. Every rank builds this list (not just rank 0) because `all_gather` writes the full set into every rank's list.

## Faded exercise 3

Complete `gather_all_values`. The `all_gather` call and the post-processing are filled. Fill in the pre-allocation of the gather list: a list of `world_size` zero tensors, each of shape `(1,)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class FakeDist:
    def __init__(self, all_values):
        self.all_values = list(all_values)
    def all_gather(self, out_list, tensor):
        for i, v in enumerate(self.all_values):
            out_list[i].copy_(t.tensor([float(v)]))


def gather_all_values(world_size, dist_module):
    gather_list = None  # TODO: fill in this step — read the prompt cell above
    dist_module.all_gather(gather_list, t.zeros(1))
    return [g.item() for g in gather_list]


vals = [2.0, 4.0, 8.0]
fd = FakeDist(vals)
result = gather_all_values(len(vals), fd)

def _test():
    vals = [2.0, 4.0, 8.0]
    fd = FakeDist(vals)
    got = gather_all_values(len(vals), fd)
    assert len(got) == 3, 'must gather one value per rank'
    assert [abs(g - v) < 1e-6 for g, v in zip(got, vals)] == [True, True, True], 'order must match rank order'

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, all_values):
        self.all_values = list(all_values)
    def all_gather(self, out_list, tensor):
        for i, v in enumerate(self.all_values):
            out_list[i].copy_(t.tensor([float(v)]))


def gather_all_values(world_size, dist_module):
    gather_list = [t.zeros(1) for _ in range(world_size)]
    dist_module.all_gather(gather_list, t.zeros(1))
    return [g.item() for g in gather_list]


vals = [2.0, 4.0, 8.0]
fd = FakeDist(vals)
result = gather_all_values(len(vals), fd)
```
</details>